In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

print(df.head())

         date  year  month  day         region    country category  \
0  2025-10-12  2025     10   12    Middle East        UAE  Chronic   
1  2020-09-15  2020      9   15     South Asia      India  Chronic   
2  2020-02-26  2020      2   26    Middle East        UAE  Vitamin   
3  2025-11-09  2025     11    9     South Asia  Sri Lanka  Chronic   
4  2022-04-04  2022      4    4  South America  Argentina  Chronic   

     medicine age_group  units_sold  unit_price  stock_level  \
0  Amlodipine      0-12         328       45.38         4774   
1  Amlodipine     26-45         371       75.49         4584   
2   Vitamin C      0-12         948       22.51         3934   
3  Amlodipine      0-12         275       63.29         4544   
4  Amlodipine     26-45         563       44.28         4284   

   expiry_days_remaining  covid_flag  
0                    466           0  
1                    181           1  
2                    556           1  
3                    330           0  

In [3]:
# create sales target
df['sales'] = df['units_sold'] * df['unit_price']
print(df[['units_sold', 'unit_price', 'sales']].head())

   units_sold  unit_price     sales
0         328       45.38  14884.64
1         371       75.49  28006.79
2         948       22.51  21339.48
3         275       63.29  17404.75
4         563       44.28  24929.64


In [4]:
# select input features
x = df[['stock_level',
        'expiry_days_remaining',
        'covid_flag']]

y = df['sales']
#We should not use too many columns for polynomial transformation because the number of features can become very large

In [6]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

print("train:", x_train.shape)
print("test:", x_test.shape)

train: (142392, 3)
test: (35598, 3)


In [8]:
# scaling feature
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [9]:
# create polynomial features
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

x_train_p = poly.fit_transform(x_train)
x_test_p = poly.transform(x_test)

print("old features:", x_train.shape[1])
print("new features:", x_train_p.shape[1])

old features: 3
new features: 9


In [10]:
# The feature name created by polynomial and interaction features
names = poly.get_feature_names_out([
    'stock',
    'expiry',
    'covid'
])

print(names)

['stock' 'expiry' 'covid' 'stock^2' 'stock expiry' 'stock covid'
 'expiry^2' 'expiry covid' 'covid^2']


In [11]:
# ridge model
ridge = Ridge(alpha=1.0)
ridge.fit(x_train_p, y_train)

pred_r = ridge.predict(x_test_p)

r2_r = r2_score(y_test, pred_r)

print("ridge r2:", r2_r)

ridge r2: 0.5312947166314237


In [13]:
# lasso model
# lasso model
lasso = Lasso(alpha=10,max_iter=20000)
lasso.fit(x_train_p, y_train)
pred_l = lasso.predict(x_test_p)

r2_l = r2_score(y_test, pred_l)

print("lasso r2:", r2_l)

lasso r2: 0.5312694286089087


In [14]:
#check mean sqr error
mse_r = mean_squared_error(y_test, pred_r)
mse_l = mean_squared_error(y_test, pred_l)

print("ridge mse:", mse_r)
print("lasso mse:", mse_l)

ridge mse: 265619195.6003392
lasso mse: 265633526.53373614


In [15]:
coef = pd.DataFrame({
    'feature': names,
    'value': lasso.coef_
})

print(coef)
#This shows the coefficients of the Lasso model. Features with coefficients close to zero have less influence on the prediction.

        feature         value
0         stock -12103.322211
1        expiry     -0.453091
2         covid   1562.637913
3       stock^2    757.846556
4  stock expiry    -35.911993
5   stock covid   -306.636210
6      expiry^2    -15.699602
7  expiry covid      0.000000
8       covid^2      0.000000


In [16]:
# count useful lasso features
n = np.sum(lasso.coef_ != 0)

print("total features:", len(lasso.coef_))
print("selected features:", n)

total features: 9
selected features: 7


In [18]:
print("ridge r2:", r2_r)
print("lasso r2:", r2_l)
print("ridge mse:", mse_r)
print("lasso mse:", mse_l)

ridge r2: 0.5312947166314237
lasso r2: 0.5312694286089087
ridge mse: 265619195.6003392
lasso mse: 265633526.53373614


In [ ]:
'''Polynomial and interaction features were generated successfully. 
Lasso regularization reduced the coefficients of less important features to zero. 
In the results, expiry covid and covid² received zero coefficients, showing that Lasso removed these features from the model. 
The remaining features contributed to the prediction of sales.'''